
Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.


Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3.6-27b")    
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.0', 'langchain': '1.3.14'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x70c0ea32c050>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x70c0ea32cd70>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [4]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.0', 'langchain': '1.3.14'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x70c0ea32c050>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x70c0ea32cd70>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'sche

In [5]:
model.invoke("Provide details about the moview Troy")

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Understand User Request**: The user asks for "details about the movie Troy". This is a straightforward request for information about the 2004 film "Troy".\n\n2.  **Identify Key Information Needed**:\n   - Title: Troy\n   - Release Year: 2004\n   - Director: Wolfgang Petersen\n   - Writers: David Benioff (based on Homer\'s Iliad)\n   - Cast: Brad Pitt, Eric Bana, Orlando Bloom, Diane Kruger, Sean Bean, Brian Cox, Peter O\'Toole, Rose Byrne, etc.\n   - Genre: Epic, Drama, War\n   - Plot Summary: Based on Homer\'s Iliad, focuses on the Trojan War, particularly the conflict between Achilles and Hector, the abduction of Helen, and the fall of Troy.\n   - Production Details: Budget, filming locations, box office performance\n   - Critical Reception: Reviews, awards/nominations\n   - Notable Aspects/Trivia: Historical/mythological accuracy, omission of gods, soundtrack, legacy\n\n3.  **Gather/Verify Facts** (from known knowled

In [7]:
response=model_with_structure.invoke("Provide details about the moview Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

Message output alongside parsed structure

In [8]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:** The user is asking for details about the movie "Inception".\n2.  **Identify Required Information:** To provide details about a movie, I need to use the `Movie` function. The function requires:\n   - `title` (string): "Inception"\n   - `year` (integer): 2010\n   - `director` (string): Christopher Nolan\n   - `rating` (number): I need to provide a rating out of 10. Inception has a high rating, typically around 8.8 on IMDB. I\'ll use 8.8.\n3.  **Verify Parameters:**\n   - Title: "Inception"\n   - Year: 2010\n   - Director: "Christopher Nolan"\n   - Rating: 8.8\n   All required parameters are available or can be reasonably inferred/known.\n4.  **Construct Function Call:**\n   ```json\n   {\n     "name": "Movie",\n     "parameters": {\n       "title": "Inception",\n       "year": 2010,\n       "director": "Christopher Nolan",\n       "rating": 8.8\n     }\n   }\n

In [16]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

# 1. Define models
class Actor(BaseModel): 
    name: str 
    role: str 

class MovieDetails(BaseModel): 
    title: str 
    year: int 
    cast: list[Actor] 
    genres: list[str] 
    budget: float | None = Field(None, description="Budget in millions USD") 

# 2. Setup Parser
parser = PydanticOutputParser(pydantic_object=MovieDetails)

# 3. Setup Prompt with format_instructions
prompt = PromptTemplate(
    template="You are a helpful assistant.\n{format_instructions}\n\nQuestion: {query}",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# 4. Chain with LCEL
chain = prompt | model | parser

# 5. Invoke
response = chain.invoke({"query": "Provide details about the movie Troy"})
print(response)

title='Troy' year=2004 cast=[Actor(name='Brad Pitt', role='Hector'), Actor(name='Eric Bana', role='Achilles'), Actor(name='Orlando Bloom', role='Paris'), Actor(name='Diane Kruger', role='Helen'), Actor(name='Sean Bean', role='King Agamemnon'), Actor(name="Peter O'Toole", role='Priam'), Actor(name='Brian Cox', role='Odysseus')] genres=['Action', 'Adventure', 'Drama'] budget=185.0
